# 15. Keep local movement and relational anomaly evidence separate

How do measured coordinate predictions become a source score without erasing movement amplitude or confusing fitting and calibration? We will reconstruct one benefit, inspect normal references and examine the frozen M, R and S channels.

## Run this notebook independently

From the repository root, install the shared environment and start Jupyter:

```bash
uv sync --project experiments/aaai
AAAI_ITERATION02_RUN_DIR=/absolute/path/to/experiments/aaai/artifacts/iteration02/PROTOCOL_HASH \
  uv run --project experiments/aaai jupyter lab experiments/aaai
```

Replace `PROTOCOL_HASH` with the exact completed run you intend to inspect. The setup displays its producer command and verifies its recorded identity before showing results. It never chooses the newest run automatically. Alternatively, put `AAAI_ITERATION02_RUN_DIR` in the repository root `.env`. Do not create another environment file. Python-dotenv calls the real functions `load_dotenv` and `find_dotenv`; the aliases below implement the requested `load_env(find_env())` spelling without printing credentials.

Python 3.12 and the notebook, plotting, data and model dependencies are declared in the existing `pyproject.toml`. Select its kernel. This notebook does not depend on cells from another notebook. A full experiment fits models through the recorded producer command; this tutorial loads those real fitted outputs, checks them and teaches how to interpret them. Missing required outputs stop execution instead of triggering synthetic examples.

A small development run has its own protocol identity. To inspect it deliberately, also set `AAAI_ITERATION02_ALLOW_DEVELOPMENT=1`. Its outputs remain labeled incomplete. Do not use that option to bypass a failed evidence verifier.

The experiment is exploratory because the available sources informed earlier work. A dataset folder label is not a diagnosis independently established here. The focal endpoint is normal versus Parkinsons, stroke, cerebral palsy and myopathic labels. The mandatory broad endpoint uses all non-normal folders. The word “normal” refers to the dataset annotation, not a clinical clearance.

In [ ]:
from pathlib import Path
import os, sys, json
from dotenv import load_dotenv as load_env, find_dotenv as find_env

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ambient").is_dir() and (p / "data/gavd").is_dir())
os.chdir(ROOT)
load_env(find_env())
BASE = ROOT / "experiments/aaai"
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from aaai_gait.iteration02_notebook_support import (
    EvidenceRun, package_versions, show_diagram, show_video, show_status,
    optional_evidence, joint_table, source_role_table, plot_counts,
    prediction_example, example_losses, plot_prediction, plot_adequacy,
    plot_timing, plot_source_scores, within_fold_metrics, plot_metric_table,
    fold_json_records, fold_tables, cohort_table, mask_bank_summary,
    measured_window, plot_paired_differences, timing_video_example, external_replication, plot_condition_confusions,
)

run = EvidenceRun.from_env()
display(run.describe().T.rename(columns={0: "value"}))
display(package_versions())
if run.mode not in {"full", "full_study"}:
    display(Markdown("**DEVELOPMENT RUN. This is incomplete evidence and cannot supply final manuscript results.**"))

## 1. Preserve two different explanations for unusual walking

A person can make small, regularly timed steps. Another person can retain an ordinary excursion but move the legs in an unusual relation. A third can use a stable compensation that makes one leg more predictable from the other. None of these examples determines a diagnosis.

M describes visible movement, including projected posture, excursion, asymmetry and velocity. R describes departure from a normal pattern of matched predictive benefits. Treat them as complementary evidence. A purely relational model can miss a small but regular movement, while a local-motion model can miss a changed relationship.

In [ ]:
show_diagram("rationale")

## 2. Compute one predictive benefit from real arrays

For the saved four-frame patch, square the difference between each coordinate prediction and the observed target, then average over the eight coordinate values. The benefit is the own-capacity error minus the both-leg error. A positive value means the other leg helped that fitted predictor for this target.

This is not a per-patient diagnostic threshold. It is one constituent observation. Its sign can vary by joint, source, support and predictor quality. Stable compensatory movement could increase rather than reduce the benefit, motivating a two-sided relational reference.

In [ ]:
row, arrays = prediction_example(run)
target = arrays["Observed target"]
own_error = float(np.mean((arrays["Own-leg correction f+h"] - target) ** 2))
both_error = float(np.mean((arrays["Both-leg correction f+g"] - target) ** 2))
display(pd.DataFrame([{"source_id": row.source_id, "own_capacity_error": own_error,
                       "both_leg_error": both_error, "predictive_benefit": own_error - both_error}]))
display(example_losses(run))

## 3. Fit M using fitting normals, not the relational reference sources

The declared local channel contains 16 descriptors. Each leg contributes knee-angle median and interquartile range, ankle horizontal and vertical excursion, and horizontal and vertical velocity variability. Four additional values describe projected trunk inclination and left-right differences. The raw clock and common scale remain intact.

M excludes local prediction error. Fit descriptor imputation, scaling and covariance on fitting-normal sources only. A projected trunk angle is not a clinical trunk examination. Camera view, depth projection and missing joints can change these descriptors, so preserve support and nuisance summaries alongside the score.

In [ ]:
display(run.table("descriptors_M.csv"))
display(pd.json_normalize(fold_json_records(run, "tabular_audit.json", field="M")))
window = measured_window(run, row)
# The cohort uses the sorted 16-joint allowlist, with x/y per frame.
joints = joint_table()["BLAZEPOSE_33 index"].tolist()
hip, knee, ankle = (window[:, joints.index(index), :2] for index in (23, 25, 27))
u, v = hip - knee, ankle - knee
norms = np.linalg.norm(u, axis=1) * np.linalg.norm(v, axis=1)
valid = np.isfinite(u).all(1) & np.isfinite(v).all(1) & (norms > 1e-8)
angles = np.degrees(np.arccos(np.clip(np.sum(u[valid] * v[valid], axis=1) / norms[valid], -1, 1)))
display(pd.DataFrame([{ "source_id": row.source_id, "window_left_knee_median_degrees":
    float(np.median(angles)) if len(angles) else np.nan, "valid_angle_frames": len(angles)}]))
print("This hand calculation is one real window. The saved M table follows the full source-aggregation rule.")
show_video(run, row)

## 4. Aggregate repeated measurements before estimating R

Repeated mask draws of the same target do not create new physical observations. Follow the frozen hierarchy: average draws for each target, then targets within windows, windows within sequences, and sequences within sources, preserving the declared joint weighting. Keep left and right constituents inspectable.

The separate reserved normal sources fit R after model and mask choices are frozen. The normal reference must match its evaluation bank. Five reference videos provide a fragile estimate, so leave-one-reference-out sensitivity is useful, but it must not select the most favorable reference after testing.

In [ ]:
display(pd.json_normalize(fold_json_records(run, "U_U/references.json")))
constituents = run.table("source_scores.csv")
constituent_columns = [name for name in constituents if name.startswith(("left_", "right_"))]
display(constituents.loc[constituents.policy.eq("U") & constituents.bank.eq("U"), ["source_id", *constituent_columns]])
display(fold_tables(run, "U_U/reference_sensitivity.csv"))

## 5. Form the fixed combined score and keep failed folds

The combined score is S = (M + R) / 2. It is fixed before inspecting abnormal outcomes. If the normal adequacy gate fails in a fold, set R to zero for every source in that fold and record the fallback. Then S = M/2 preserves the M ranking within that fold.

Do not exclude difficult sources because the relation failed. A fallback result can still be evaluated, but it does not establish relational learning. The primary detection comparison is S against M; the masking-policy contrasts are separate questions.

In [ ]:
scores = run.table("source_scores.csv", columns=("outer_fold", "source_id", "policy", "bank", "M", "R", "S", "fallback"))
primary_rows = scores.bank.isin(["U", "E_U"])
assert np.isfinite(scores.loc[primary_rows, ["M", "R", "S"]].to_numpy(float)).all()
available = np.isfinite(scores[["M", "R", "S"]].to_numpy(float)).all(axis=1)
assert np.allclose(scores.loc[available, "S"], (scores.loc[available, "M"] + scores.loc[available, "R"]) / 2), "Saved scores do not follow the frozen formula."
fallback = scores.fallback.astype(str).str.lower().isin(["true", "1"])
assert np.allclose(scores.loc[fallback & available, "R"], 0)
print("Unavailable diagnostic score rows:", int((~available).sum()))
display(scores.groupby(["policy", "bank", "outer_fold"]).agg(sources=("source_id", "nunique"), fallback_rows=("fallback", "sum")))

## 6. Inspect score distributions without replacing the primary estimator

The empirical curves below describe the saved scores. A pooled plot can look separated because different folds use slightly different score scales. It is not the primary AUROC calculation. The next notebook computes rankings within each fold and combines them using fixed normal-by-abnormal pair counts.

Do not reverse R's direction or retune the M/R mixture after looking at these curves. Inspect sham or nuisance comparators as explanations of a result, not as opportunities to find a favorable score.

In [ ]:
primary_bank = "U" if "U" in scores.bank.unique() else "E_U"
display(plot_source_scores(run, policy="U", bank=primary_bank)); plt.close("all")
display(scores.loc[scores.policy.eq("U") & scores.bank.eq(primary_bank), ["source_id", "M", "R", "S", "S_sham", "nuisance_distance", "nuisance_supervised"]])

## Preserve the evidence trail

The run directory is the unit of reproducibility. Keep its input hashes, source roles, fitted models, score references, target predictions, source scores, verification record and failed-comparison ledger together. A successful notebook execution establishes that this tutorial can reproduce its displayed analysis. It does not establish a neurological cause or a clinically valid classifier.

The [iteration02 protocol](docs/iteration02/protocol.md), [results](docs/iteration02/results.md), [limitations](docs/iteration02/limitations.md), and [draft02 readiness review](docs/draft02/README.md) must agree with the same run. The earlier draft01 and notebooks 00–11 remain historical evidence. New figures are written as SVG and PDF masters with PNG previews and an input-hash sidecar under this run's `figures/notebooks` directory.

In [ ]:
record_path = run.record('15_local_and_relational_anomaly_scores')
print("Recorded the exact real artifacts read by this notebook:", record_path)
print("This notebook did not replace the experiment producer or alter its source roles.")